## Flow chart : Graphique du flux de patients
Consigne : Calculer les effectifs permettant de compléter le graphique avec les flux de patients (patients' flow chart) selon le modèle proposé

Sujets choisis durant tout le travaille: nous considérons l'ensemble des patients, c'est-à-dire à la fois les hospitalisés du dossier CTN001 (ascii-data-files_nida-ctn-0001) et du dossier CTN002 (sujets ambulatoire : ascii-data-files_nida-ctn-0002), que nous pouvons retrouvé dans "les documents disponibles"-> afin d'avoir des tailles d'échantillons plus importantes.

À la suite de l’analyse du flowchart à compléter, nous recherchons les fichiers contenant les informations suivantes :
- Les sujets inclus et non inclus
- Les sujets randomisés selon les bras suivants : échantillon BUP/NL et échantillon Clonidine
- Les raisons d’exclusion
- Les sujets que les chercheurs ont l’intention de traiter (analyse en ITT)
- Les sujets effectivement traités (analyse per protocole) »

Dans les fichiers : CTN001 et CTN002, nous observons que les données qui nous intéresse se situe dans les fichiers : ds.csv et dm.csv  

In [1]:
import polars as pl  

In [2]:
dm1 = pl.read_csv("dm1.csv")
dm2 = pl.read_csv("dm2.csv")

In [3]:
dm = pl.concat([dm1, dm2])

In [4]:
#Vérification des données 
dm.sample(4)

STUDYID,DOMAIN,USUBJID,EPOCH,VISIT,VISITNUM,RFSTDTC,RFENDTC,SITEID,BRTHDTC,AGE,AGEU,SEX,RACE,ETHNIC,ARMCD,ARM,COUNTRY,DMDTC,DMDY
str,str,str,str,str,i64,i64,i64,str,i64,str,str,str,str,str,str,str,str,i64,str
"""NIDA-CTN-0002""","""DM""","""02_019588""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,2001,null,1963,"""38.622861054""","""YEARS""","""M""","""OTHER""",null,"""BUPNAL""","""BUPRENORPHINE/NALOXONE""","""USA""",2001,"""1"""
"""NIDA-CTN-0001""","""DM""","""01_043642""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2002,2002,null,1971,"""30.579055441""","""YEARS""","""M""","""SPANISH, HISPANIC, OR LATINO""","""HISPANIC""","""BUPNAL""","""BUPRENORPHINE/NALOXONE""","""USA""",2002,"""-3"""
"""NIDA-CTN-0001""","""DM""","""01_027452""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2002,2002,null,1962,"""39.400410678""","""YEARS""","""M""","""WHITE""",null,"""CLON""","""CLONIDINE""","""USA""",2002,"""-7"""
"""NIDA-CTN-0002""","""DM""","""02_029448""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,2001,null,1975,"""25.629021218""","""YEARS""","""F""","""SPANISH, HISPANIC, OR LATINO""","""HISPANIC""","""BUPNAL""","""BUPRENORPHINE/NALOXONE""","""USA""",2001,"""-13"""


In [5]:
# Sélection les colonnes pertinant pour faciliter la visualisation
dm = dm.select(
    pl.col('USUBJID'), #Identifiant 
    pl.col('EPOCH'), #Statue du sujet : ici SCREENING 
    pl.col('ARM'), # bras de traitemEnt 
    pl.col('ARMCD'), # annonation du bras de traitement
)
dm.sample(3)

USUBJID,EPOCH,ARM,ARMCD
str,str,str,str
"""02_058028""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_018478""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_050446""","""ACTIVE""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""


In [6]:
dm.null_count().sum_horizontal().sum() # pas de valeurs manquantes

0

In [7]:
#Vérifier la redondance entre les colonnes "ARM" et "ARMCD"
redon = dm.select(["ARM", "ARMCD"]).with_columns(
    (
        ((pl.col("ARM") == "BUPRENORPHINE/NALOXONE") & (pl.col("ARMCD") == "BUPNAL")) |
        ((pl.col("ARM") == "CLONIDINE") & (pl.col("ARMCD") == "CLON")) |
        ((pl.col("ARM") == "SCREEN FAILURE") & (pl.col("ARMCD") == "SCRFAIL"))
    ).alias("correct_mapping")
)

incorrect = redon.filter(~pl.col("correct_mapping")) # permet d'afficher toutes les lignes où la correspondance n’est pas correcte.
incorrect

ARM,ARMCD,correct_mapping
str,str,bool


In [8]:
# Vérifier le nombre de patient unique 
print((dm.group_by("USUBJID").len().filter(pl.col("len") > 1).height))

0


In [9]:
dm.shape

(411, 4)

Nous pouvons observer un total de 411 participants ayant postulé pour l’expérience.

In [10]:
dmIncluded = dm.filter(pl.col("ARM") != "SCREEN FAILURE")
dmIncluded

USUBJID,EPOCH,ARM,ARMCD
str,str,str,str
"""01_000579""","""SCREENING""","""CLONIDINE""","""CLON"""
"""01_001362""","""SCREENING""","""CLONIDINE""","""CLON"""
"""01_001490""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""01_002199""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""01_002844""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
…,…,…,…
"""02_098074""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_098425""","""SCREENING""","""CLONIDINE""","""CLON"""
"""02_099053""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""


En excluant les sujets n'ayant pas pu être retenue (68 sujets), nous comptons 343 participants au total cochant les critères de juguement pour participer à l'expérience. 

In [11]:
# Recherche : Bras de traitement : échantillon BUX/NL et Clonidine
dmIncluded["ARM"].value_counts()

ARM,count
str,u32
"""BUPRENORPHINE/NALOXONE""",233
"""CLONIDINE""",110


u32 : aucune valeur négative signalée.

Suite à la randomisation des 343 patients retenus :
- L’échantillon BUP/NC a été attribué à 233 participants.
- L’échantillon CLO a été attribué à 110 participants.

110 + 233 = 343 participants

L’analyse en intention de traiter (ITT) consiste à inclure dans l’analyse de l’effet du traitement sur le critère de jugement tous les patients randomisés, dans le groupe où ils ont été randomisés, sans tenir compte des évènements intercurrents qui auraient pu survenir : erreur de traitement, arrêt du traitement (treatment discontinuation, treatment stopped prematurely ), retrait de l’étude (withdrawal ), recours au traitement de l’autre groupe, inclusion à tort (included in error), patient perdu de vue (lost to follow-up)

référence : https://sfpt-fr.org/livreblancmethodo/part5/file_6.htm

Nous recherchons donc les sujets ayant été exclus à la suite d’un événement intercurrent pour les deux bras de traitement. C'est dans le fichier ds.csv que nous pouvons observer les raisons de cette exclusion.

In [12]:
ds1 = pl.read_csv("ds1.csv")
ds2 = pl.read_csv("ds2.csv")

In [13]:
ds1 = ds1.select(sorted(ds1.columns))
ds2 = ds2.select(sorted(ds2.columns))

In [14]:
ds = pl.concat([ds1, ds2])
ds

DOMAIN,DSCAT,DSDECOD,DSDTC,DSDY,DSOCCUR,DSSEQ,DSSTDTC,DSSTDY,DSTERM,EPOCH,STUDYID,USUBJID,VISIT,VISITNUM
str,str,str,i64,str,str,i64,i64,str,str,str,str,str,str,i64
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT WITHDREW FROM STUD…",2001,"""14""","""Y""",1,2001,"""3""","""PARTICIPANT WITHDREW FROM STUD…","""ACTIVE""","""NIDA-CTN-0001""","""01_000579""","""STUDY DAY 14""",14
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT WITHDREW FROM STUD…",2001,"""4""","""Y""",1,2001,"""4""","""PARTICIPANT WITHDREW FROM STUD…","""ACTIVE""","""NIDA-CTN-0001""","""01_001362""","""STUDY DAY 4""",4
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT WITHDREW FROM STUD…",2001,"""56""","""Y""",2,2001,"""4""","""PARTICIPANT WITHDREW FROM STUD…","""ACTIVE""","""NIDA-CTN-0001""","""01_001362""","""FOLLOW-UP 1""",15
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT WITHDREW FROM STUD…",2002,"""93""","""Y""",3,2001,"""4""","""PARTICIPANT WITHDREW FROM STUD…","""ACTIVE""","""NIDA-CTN-0001""","""01_001362""","""FOLLOW-UP 2""",16
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT WITHDREW FROM STUD…",2002,"""211""","""Y""",4,2001,"""4""","""PARTICIPANT WITHDREW FROM STUD…","""ACTIVE""","""NIDA-CTN-0001""","""01_001362""","""FOLLOW-UP 3""",17
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""DS""","""DISPOSITION EVENT""","""VOLUNTARY TRANSFERRED TO ANOTH…",2001,"""112""","""Y""",5,null,""" ""","""VOLUNTARY TRANSFERRED TO ANOTH…","""ACTIVE""","""NIDA-CTN-0002""","""02_099926""","""FOLLOW-UP 2""",16
"""DS""","""DISPOSITION EVENT""","""VOLUNTARY TRANSFERRED TO ANOTH…",2001,"""198""","""Y""",8,null,""" ""","""VOLUNTARY TRANSFERRED TO ANOTH…","""ACTIVE""","""NIDA-CTN-0002""","""02_099926""","""FOLLOW-UP 3""",17
"""DS""","""DISPOSITION EVENT""","""VOLUNTARY TRANSFERRED TO ANOTH…",2001,"""51""","""Y""",3,null,""" ""","""VOLUNTARY TRANSFERRED TO ANOTH…","""ACTIVE""","""NIDA-CTN-0002""","""02_099926""","""FOLLOW-UP 1""",15


In [15]:
ds1.columns

['DOMAIN',
 'DSCAT',
 'DSDECOD',
 'DSDTC',
 'DSDY',
 'DSOCCUR',
 'DSSEQ',
 'DSSTDTC',
 'DSSTDY',
 'DSTERM',
 'EPOCH',
 'STUDYID',
 'USUBJID',
 'VISIT',
 'VISITNUM']

In [16]:
ds2.columns

['DOMAIN',
 'DSCAT',
 'DSDECOD',
 'DSDTC',
 'DSDY',
 'DSOCCUR',
 'DSSEQ',
 'DSSTDTC',
 'DSSTDY',
 'DSTERM',
 'EPOCH',
 'STUDYID',
 'USUBJID',
 'VISIT',
 'VISITNUM']

In [17]:
ds1.shape

(385, 15)

In [18]:
ds2.shape

(1107, 15)

Nous remarquons que 'DSOCCUR' et 'DSDECOD' ne sont pas dans le même ordre. Polars considère cela comme une différence dans les colonnes.
Nous mettons donc les variables de nos deux DataFrames en ordre alphabétique

In [19]:
ds1 = ds1.select(sorted(ds1.columns)) #sorted : permet un tri alphabétique de nos variables
ds2 = ds2.select(sorted(ds2.columns))

In [20]:
ds = pl.concat([ds1, ds2])

In [21]:
#vérification
ds.sample(4)

DOMAIN,DSCAT,DSDECOD,DSDTC,DSDY,DSOCCUR,DSSEQ,DSSTDTC,DSSTDY,DSTERM,EPOCH,STUDYID,USUBJID,VISIT,VISITNUM
str,str,str,i64,str,str,i64,i64,str,str,str,str,str,str,i64
"""DS""","""DISPOSITION EVENT""","""VOLUNTARY TRANSFERRED TO ANOTH…",2002,"""91""","""Y""",4,null,""" ""","""VOLUNTARY TRANSFERRED TO ANOTH…","""ACTIVE""","""NIDA-CTN-0002""","""02_045101""","""FOLLOW-UP 2""",16
"""DS""","""DISPOSITION EVENT""","""VOLUNTARY TRANSFERRED TO ANOTH…",2002,"""193""","""Y""",8,2001,"""14""","""VOLUNTARY TRANSFERRED TO ANOTH…","""ACTIVE""","""NIDA-CTN-0001""","""01_091767""","""FOLLOW-UP 3""",17
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT COMPLETED ACTIVE P…",2001,"""14""","""Y""",1,2001,"""14""","""PARTICIPANT COMPLETED ACTIVE P…","""ACTIVE""","""NIDA-CTN-0002""","""02_033087""","""STUDY DAY 14""",14
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT NO LONGER ATTENDS …",2001,"""14""","""Y""",1,2001,"""9""","""PARTICIPANT NO LONGER ATTENDS …","""ACTIVE""","""NIDA-CTN-0002""","""02_030892""","""STUDY DAY 14""",14


In [22]:
ds.columns

['DOMAIN',
 'DSCAT',
 'DSDECOD',
 'DSDTC',
 'DSDY',
 'DSOCCUR',
 'DSSEQ',
 'DSSTDTC',
 'DSSTDY',
 'DSTERM',
 'EPOCH',
 'STUDYID',
 'USUBJID',
 'VISIT',
 'VISITNUM']

In [23]:
ds.select(['EPOCH']).unique()

EPOCH
str
"""SCREENING"""
"""ACTIVE"""


In [24]:
#Selectionne nos colonnes pertinants
ds_n = ds.select(
    pl.col('USUBJID'), #identifiant
    pl.col('DSDECOD'), #raison de l'exclusion
    pl.col('EPOCH'), # stade su sujet : SCREENING ou ACTIVE
    pl.col('VISITNUM'),# jour de visite 
)
ds_n.sample(4)



USUBJID,DSDECOD,EPOCH,VISITNUM
str,str,str,i64
"""02_008499""","""PARTICIPANT COMPLETED ACTIVE P…","""ACTIVE""",15
"""02_033087""","""PARTICIPANT COMPLETED ACTIVE P…","""ACTIVE""",17
"""01_086738""","""DECLINED STUDY PARTICIPATION""","""SCREENING""",0
"""02_070154""","""PARTICIPANT COMPLETED ACTIVE P…","""ACTIVE""",15


In [25]:
#Filtrage de nos sujets ayant été exclus lors de la phase active et les raisons de cette exclusion
dsDischarged = (
    ds_n
    .filter(pl.col("EPOCH") != "SCREENING")
    .with_columns((pl.col("VISITNUM") < 14).alias("avant_et_active"))
    .filter(
        pl.col("avant_et_active") & 
        (pl.col("DSDECOD") != "PARTICIPANT COMPLETED ACTIVE PHASE OF STUDY")
    )
    .select(["USUBJID", "DSDECOD", "VISITNUM", "avant_et_active"])
    .unique()
)

dsDischarged

USUBJID,DSDECOD,VISITNUM,avant_et_active
str,str,i64,bool
"""02_013437""","""PARTICIPANT WITHDREW FROM STUD…",3,true
"""02_045260""","""VOLUNTARY TRANSFERRED TO ANOTH…",3,true
"""02_031720""","""PARTICIPANT WITHDREW FROM STUD…",10,true
"""01_003653""","""PARTICIPANT WITHDREW FROM STUD…",10,true
"""01_001362""","""PARTICIPANT WITHDREW FROM STUD…",4,true
…,…,…,…
"""01_018599""","""VOLUNTARY TRANSFERRED TO ANOTH…",3,true
"""02_077812""","""PARTICIPANT WITHDREW FROM STUD…",2,true
"""01_018076""","""ADMINISTRATIVELY WITHDRAWN""",1,true


In [26]:
dsDischarged["VISITNUM"].value_counts()

VISITNUM,count
i64,u32
4,5
7,2
10,8
13,1
1,6
5,1
8,4
2,10
3,21


In [27]:
dsDischarged.select(pl.col("DSDECOD").value_counts())

DSDECOD
struct[2]
"{""PARTICIPANT NO LONGER ATTENDS CLINIC"",4}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM-THERAPEUTIC COMMUNITY"",2}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM (INDICATE TYPE)"",13}"
"{""INPATIENT DETOX OR TREATMENT"",3}"
"{""PARTICIPANT HAS DEVELOPED SENSITIVITY OR ALLERGY TO BUPRENORPHINE/NALOXONE OR CLONIDINE"",1}"
…
"{""PARTICIPANT WITHDREW FROM STUDY"",17}"
"{""HOSPITALIZED OR DEVELOPED ACUTE MEDICAL CONDITION WHICH WOULD MAKE FURTHER TREATMENT HAZARDOUS"",2}"
"{""PARTICIPANT IS IN A CONTROLLED ENVIRONMENT"",2}"


Nous joignons les variables "dmIncluded" et "dsDischarged" afin de connaître les raisons de chaque exclusion pour nos deux bras de traitement.

In [28]:
join_df = dmIncluded.join(dsDischarged, on="USUBJID", how="right")
join_df.sample(3)

EPOCH,ARM,ARMCD,USUBJID,DSDECOD,VISITNUM,avant_et_active
str,str,str,str,str,i64,bool
"""SCREENING""","""CLONIDINE""","""CLON""","""01_018599""","""VOLUNTARY TRANSFERRED TO ANOTH…",3,true
null,null,null,"""02_051268""","""HOSPITALIZED OR DEVELOPED ACUT…",4,true
"""SCREENING""","""CLONIDINE""","""CLON""","""01_096578""","""PARTICIPANT WITHDREW FROM STUD…",5,true


In [29]:
#Vérification
join_df.shape

(58, 7)

Nous retrouvons bien 58 sujets exclus après avoir joint nos deux fichiers.

In [30]:
# Colonne qui nous intéresse
exlu = join_df.select(
    pl.col("ARMCD"),
    pl.col("DSDECOD")
)

In [31]:
# Filtrer les sujets exclus ayant le traitement CLO
exclu_clon = exlu.filter(pl.col("ARMCD") == "CLON")
exclu_clon

ARMCD,DSDECOD
str,str
"""CLON""","""VOLUNTARY TRANSFERRED TO ANOTH…"
"""CLON""","""PARTICIPANT WITHDREW FROM STUD…"
"""CLON""","""VOLUNTARY TRANSFERRED TO ANOTH…"
"""CLON""","""PARTICIPANT WITHDREW FROM STUD…"
"""CLON""","""VOLUNTARY TRANSFERRED TO ANOTH…"
…,…
"""CLON""","""VOLUNTARY TRANSFERRED TO ANOTH…"
"""CLON""","""VOLUNTARY TRANSFERRED TO ANOTH…"
"""CLON""","""PARTICIPANT WITHDREW FROM STUD…"


In [32]:
exclu_clon.select(pl.col("DSDECOD").value_counts())

DSDECOD
struct[2]
"{""PARTICIPANT WITHDREW FROM STUDY"",10}"
"{""ADMINISTRATIVELY WITHDRAWN"",1}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM-THERAPEUTIC COMMUNITY"",1}"
"{""PARTICIPANT IS IN A CONTROLLED ENVIRONMENT"",1}"
"{""INPATIENT DETOX OR TREATMENT"",1}"
"{""PARTICIPANT HAS DEVELOPED SENSITIVITY OR ALLERGY TO BUPRENORPHINE/NALOXONE OR CLONIDINE"",1}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM-METHADONE"",7}"
"{""PARTICIPANT NO LONGER ATTENDS CLINIC"",1}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM (INDICATE TYPE)"",8}"


Durant la phase active du traitement CLON, 31 participant ont été exluent de l'expérience pour les raisons suivant : 
- Transfert volontaire vers un autre programme de traitement : 8
- Transfert volontaire vers un autre programme de traitement - méthadone : 7
- Le participant se trouve dans un environnement contrôlé : 1
- Transfert volontaire vers un autre programme de traitement - communauté thérapeutique : 1
- Le participant ne se présente plus à la clinique : 1
- Le participant s’est retiré de l’étude : 10
- Désintoxication ou traitement en hospitalisation : 1
- Le participant a développé une sensibilité ou une allergie à la buprénorphine/naloxone ou à la clonidine : 1
- Retiré administrativement : 1

Calcule : 8+7+1+1+1+10+1+1+1= 31

In [33]:
# Même procédé mais cette fois ci avec le traitement BUX/NL
exclu_bupnal = exlu.filter(pl.col("ARMCD") == "BUPNAL")
exclu_bupnal

ARMCD,DSDECOD
str,str
"""BUPNAL""","""PARTICIPANT WITHDREW FROM STUD…"
"""BUPNAL""","""PARTICIPANT WITHDREW FROM STUD…"
"""BUPNAL""","""PARTICIPANT WITHDREW FROM STUD…"
"""BUPNAL""","""VOLUNTARY TRANSFERRED TO ANOTH…"
"""BUPNAL""","""HOSPITALIZED OR DEVELOPED ACUT…"
…,…
"""BUPNAL""","""VOLUNTARY TRANSFERRED TO ANOTH…"
"""BUPNAL""","""INPATIENT DETOX OR TREATMENT"""
"""BUPNAL""","""PARTICIPANT NO LONGER ATTENDS …"


In [34]:
exclu_bupnal.select(pl.col("DSDECOD").value_counts())

DSDECOD
struct[2]
"{""HOSPITALIZED OR DEVELOPED ACUTE MEDICAL CONDITION WHICH WOULD MAKE FURTHER TREATMENT HAZARDOUS"",1}"
"{""PARTICIPANT WITHDREW FROM STUDY"",7}"
"{""DISCHARGED FOR OTHER (SPECIFY)"",1}"
"{""PARTICIPANT IS IN A CONTROLLED ENVIRONMENT"",1}"
"{""PARTICIPANT NO LONGER ATTENDS CLINIC"",3}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM (INDICATE TYPE)"",5}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM-THERAPEUTIC COMMUNITY"",1}"
"{""ADMINISTRATIVELY WITHDRAWN"",4}"
"{""VOLUNTARY TRANSFERRED TO ANOTHER TREATMENT PROGRAM-METHADONE"",1}"


Durant la phase active du traitement BUPNAL, 26 participant ont été exluent de l'expérience pour les raisons suivant : 
- Déchargé pour autre raison : 1
- Transfert volontaire vers un autre programme de traitement - méthadone : 1
- Le participant ne se présente plus à la clinique : 3
- Retiré administrativement : 4
- Désintoxication ou traitement en hospitalisation : 2
- Le participant se trouve dans un environnement contrôlé : 1
- Hospitalisé ou a développé une condition médicale aiguë rendant le traitement dangereux : 1
- Transfert volontaire vers un autre programme de traitement : 5
- Transfert volontaire vers un autre programme de traitement - communauté thérapeutique :1
- Le participant s’est retiré de l’étude : 7

Calcule : 1+1+3+4+2+1+1+5+1+7 = 26

Ainsi pour l’analyse en intention de traiter : 
- Traitement CLON = 233 - 26 = 207
- Traitement BUP/NAL = 110 - 31 + 79

L’analyse per protocole porte uniquement sur les données des participants à un essai clinique qui ont achevé la totalité du plan de traitement et qui ont parfaitement respecté les instructions du protocole d’essai. Il s’agit par conséquent uniquement d’une analyse de l’effet d’un traitement, tel qu’un médicament, dans une situation « idéale ».

Nous comprenons dans ce paragraphe que, pour l’analyzed per protocole, le sujet doit avoir rempli tous les critères d’évaluation (phase active), respecté les instructions du protocole d’essai pendant la période de test et être présent après 14 jours.
'

référence : https://toolbox.eupati.eu/glossary/analyse-per-protocole/?lang=fr

In [35]:
ds.sample(3)

DOMAIN,DSCAT,DSDECOD,DSDTC,DSDY,DSOCCUR,DSSEQ,DSSTDTC,DSSTDY,DSTERM,EPOCH,STUDYID,USUBJID,VISIT,VISITNUM
str,str,str,i64,str,str,i64,i64,str,str,str,str,str,str,i64
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT COMPLETED ACTIVE P…",2001,"""90""","""Y""",3,null,""" ""","""PARTICIPANT COMPLETED ACTIVE P…","""ACTIVE""","""NIDA-CTN-0002""","""02_086880""","""FOLLOW-UP 2""",16
"""DS""","""DISPOSITION EVENT""","""PARTICIPANT NO LONGER ATTENDS …",2002,"""210""","""Y""",4,2001,"""14""","""PARTICIPANT NO LONGER ATTENDS …","""ACTIVE""","""NIDA-CTN-0001""","""01_047430""","""FOLLOW-UP 3""",17
"""DS""","""DISPOSITION EVENT""","""DISCHARGED FOR OTHER (SPECIFY)""",2001,"""14""","""Y""",3,2001,"""14""","""DISCHARGED FOR OTHER (SPECIFY)""","""ACTIVE""","""NIDA-CTN-0002""","""02_026834""","""STUDY DAY 14""",14


In [36]:
dsCompleted = (
    ds.filter(
        (pl.col("EPOCH") == "ACTIVE") &
        (pl.col("VISITNUM") >= 14) &
        (pl.col("DSDECOD") == "PARTICIPANT COMPLETED ACTIVE PHASE OF STUDY")
    )
    .select("USUBJID")
    .unique()
)

dsCompleted

USUBJID
str
"""01_045322"""
"""02_036584"""
"""02_081773"""
"""01_096713"""
"""02_044005"""
…
"""01_034039"""
"""02_098425"""
"""02_078658"""


Un total de 208 participants a réussi à compléter l’étude.

In [37]:
join_full= dsCompleted.join(dmIncluded, on="USUBJID", how="left")
join_full.sample(3)

USUBJID,EPOCH,ARM,ARMCD
str,str,str,str
"""02_091708""","""SCREENING""","""CLONIDINE""","""CLON"""
"""01_044124""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_000549""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""


In [38]:
#Vérification 
join_full.shape

(208, 4)

In [39]:
# Sujets ayant complété l'étude selon le bras de traitement : CLON
full_clon = join_full.filter(pl.col("ARMCD") == "CLON")
full_clon

USUBJID,EPOCH,ARM,ARMCD
str,str,str,str
"""02_081773""","""ACTIVE""","""CLONIDINE""","""CLON"""
"""02_043111""","""SCREENING""","""CLONIDINE""","""CLON"""
"""01_077274""","""SCREENING""","""CLONIDINE""","""CLON"""
"""02_050489""","""SCREENING""","""CLONIDINE""","""CLON"""
"""02_091708""","""SCREENING""","""CLONIDINE""","""CLON"""
…,…,…,…
"""01_018599""","""SCREENING""","""CLONIDINE""","""CLON"""
"""02_007784""","""SCREENING""","""CLONIDINE""","""CLON"""
"""01_020369""","""SCREENING""","""CLONIDINE""","""CLON"""


In [40]:
# Sujets ayant complété l'étude selon le bras de traitement : BUPNAL
full_bupnal = join_full.filter(pl.col("ARMCD") == "BUPNAL")
full_bupnal

USUBJID,EPOCH,ARM,ARMCD
str,str,str,str
"""01_045322""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_036584""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""01_096713""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_044005""","""ACTIVE""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_059257""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
…,…,…,…
"""01_027631""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""01_034039""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""
"""02_078658""","""SCREENING""","""BUPRENORPHINE/NALOXONE""","""BUPNAL"""


Pour conclure le flowchart, après l’analyse des patients ayant rempli tous les critères d’évaluation, respecté les instructions du protocole d’essai pendant la période de test et été présents après 14 jours, nous obtenons les résultats suivants :
- BUP/NAL : 172 participants
- CLON : 36 participants

Visualisation Flowchart : 